In [3]:
from langchain.messages import RemoveMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import before_model
from langgraph.runtime import Runtime
from langchain_core.runnables import RunnableConfig
from typing import Any
import os

os.environ["DEEPSEEK_API_KEY"] = "sk-49892f84100b4c73843a82e371fc3522"


@before_model
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Keep only the last few messages to fit context window."""
    messages = state["messages"]

    if len(messages) <= 3:
        return None  # No changes needed

    first_msg = messages[0]
    recent_messages = messages[-3:] if len(messages) % 2 == 0 else messages[-4:]
    new_messages = [first_msg] + recent_messages

    return {
        "messages": [
            RemoveMessage(id=REMOVE_ALL_MESSAGES),
            *new_messages
        ]
    }


agent = create_agent(
    "deepseek-flash",
    middleware=[trim_messages],
    checkpointer=InMemorySaver(),
)

config: RunnableConfig = {"configurable": {"thread_id": "1"}}

agent.invoke({"messages": "hi, my name is bob"}, config)
agent.invoke({"messages": "write a short poem about cats"}, config)
agent.invoke({"messages": "now do the same but for dogs"}, config)
final_response = agent.invoke({"messages": "what's my name?"}, config)

[message.content for message in final_response["messages"]]

['hi, my name is bob',
 'For Bob—\n\nSoft paws tread on silent floors,\nWhiskers twitch at closing doors.\nThey curl in sun, a sleepy heap,\nThen chase the shadows while we sleep.\nWith eyes like moons and tails that sway,\nThey own the night, then own the day.',
 'now do the same but for dogs',
 'For Bob—\n\nWarm wet nose and wagging tail,\nGreeting you without fail.\nA ball, a stick, a joyful bound,\nThe happiest friend you’ve ever found.\nThrough every season, come what may,\nThey love you more than words can say.',
 "what's my name?",
 'Your name is Bob.']